<div style="padding: 20px; background: linear-gradient(90deg, #FDC830 0%, #F37335 100%); border-radius: 10px; color: white;">
    <h1 style="color: white; border-bottom: none;">🔨 Module 8.2: RAG as a Tool for Agents</h1>
    <p style="font-size: 1.2em; opacity: 0.9;">Giving the LLM the ability to search databases autonomously.</p>
</div>

---

## 1. Custom Retrievers via `@tool`

In the modern LangChain/LangGraph ecosystem, an Agent interacts with the world via `Tools`. 
We can take any standard Vector Search function and wrap it in a `@tool` decorator. 
Crucially, we must give the tool a **Description** (via the docstring). The LLM reads the Description to figure out whether it should use the tool or not!

## 2. Multi-Tool Agents
We are going to give our Agent TWO different Vector Databases:
1. An AI Models Database.
2. A Programming Languages Database.

We will ask it a complex question, and watch it use BOTH databases to formulate an answer.

In [ ]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from dotenv import load_dotenv
import os
import warnings

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
load_dotenv()

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# --- Database 1: AI Models ---
ai_docs = [
    Document(page_content="Llama 3 is an open-weights model developed by Meta."),
    Document(page_content="Claude 3.5 Sonnet is Anthropic's flagship coding model."),
    Document(page_content="Gemini 1.5 Pro features a massive 1M+ token context window."),
]
ai_vs = Chroma.from_documents(ai_docs, embeddings, collection_name="agent_ai_db_v5")

@tool
def ai_models_search(query: str) -> str:
    """Search for information exclusively about AI language models like Llama, Claude, and Gemini."""
    docs = ai_vs.similarity_search(query, k=1)
    return "\n".join([d.page_content for d in docs])

# --- Database 2: Programming ---
code_docs = [
    Document(page_content="Python relies heavily on the Global Interpreter Lock (GIL) for concurrency."),
    Document(page_content="Rust ensures memory safety without a garbage collector via Ownership."),
    Document(page_content="Go handles concurrency seamlessly using Goroutines."),
]
code_vs = Chroma.from_documents(code_docs, embeddings, collection_name="agent_code_db_v5")

@tool
def programming_search(query: str) -> str:
    """Search for information exclusively about programming languages like Python, Rust, and Go."""
    docs = code_vs.similarity_search(query, k=1)
    return "\n".join([d.page_content for d in docs])

# Give the agent both tools!
tools = [ai_models_search, programming_search]
print("Databases and Custom Tools created.")

## 3. Initializing the LangGraph ReAct Agent
We will use LangGraph's `create_react_agent`, which natively supports tool-calling capabilities of modern LLMs. It creates a complete StateGraph behind the scenes that loops between the LLM node and a Tools Execution node until the answer is complete.

In [ ]:
groq_api_key = os.environ.get("GROQ_API_KEY")

if groq_api_key:
    # Note: Tool calling agents require a very capable model.
    llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
    
    # Create the ReAct agent graph
    app = create_react_agent(llm, tools=tools)
    
    print("\n--- STARTING AGENTIC RAG MISSION ---\n")
    # Ask a question that requires BOTH databases to answer!
    query = "What is the context window of Gemini 1.5?"
    
    result = app.invoke({"messages": [("user", query)]})
    print("\n✨ FINAL AGENT ANSWER ✨")
    print(result["messages"][-1].content)
else:
    print("GROQ_API_KEY missing. Please add it to your .env file.")